# A scratchpad Notebook 

Useful to develop new functionalities, test existing code, learn Python, run generated examples, etc

In [31]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks


from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)


%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
from src.utils.config_mngr import global_config
from langchain_postgres import PGEngine

pgconf = global_config().merge_with("config/components/pgvector.yaml").get_dict("default_local_container")

connection_string = (
    f"postgresql+asyncpg://{pgconf['postgres_user']}:{pgconf['postgres_password']}@{pgconf['postgres_host']}"
    f":{pgconf['postgres_port']}/{pgconf['postgres_db']}"
)

pg_engine = PGEngine.from_connection_string(url=connection_string)

In [ ]:
from src.ai_core.embeddings import EmbeddingsFactory
from src.ai_core.vector_store import VectorStoreFactory


factory = VectorStoreFactory(
    id="PgVector",
    embeddings_factory=EmbeddingsFactory(embeddings_id="embeddings_768_fake"),
    chroma_collection_name="my_documents",
)

vector_store = factory.vector_store

2025-07-30 18:23:05.237 | DEBUG    | src.ai_core.vector_store:_create_pg_vector_store:24 - Use existing pgvector table : vectorstore_embeddings_768
2025-07-30 18:23:05.254 | INFO     | src.ai_core.vector_store:vector_store:207 - get vector store  : PgVector/my_documents_embeddings_768_fake cache_embeddings: False


In [ ]:
# factory.clean()

In [35]:
from langchain_core.documents import Document


vector_store.add_documents(documents=[Document(page_content="example")])
vector_store.add_documents(documents=[Document(page_content="another")])

['b4f3a866-2a50-40ab-acc5-863c5d27f1be']

In [36]:
vector_store.similarity_search("hello")

[Document(id='a63974e7-0e42-4dca-9983-75da06cb82d4', metadata={'len': None, 'other_data': None}, page_content='another'),
 Document(id='dcd9eed6-c856-4d5a-b0b4-7b3c6d5e2a22', metadata={'len': None, 'other_data': None}, page_content='another'),
 Document(id='fdc3e577-dd4e-40c3-b3af-e371005b2145', metadata={'len': None, 'other_data': None}, page_content='another'),
 Document(id='6d65cf9e-50a5-45f7-9964-e0c2faf7f230', metadata={'len': None, 'other_data': None}, page_content='another')]

In [ ]:
vector_store2 = VectorStoreFactory(
    id="PgVector",
    embeddings_factory=EmbeddingsFactory(embeddings_id="embeddings_768_fake"),
    chroma_collection_name="my_documents",
).vector_store

2025-07-30 18:23:05.449 | DEBUG    | src.ai_core.vector_store:_create_pg_vector_store:24 - Use existing pgvector table : vectorstore_embeddings_768
2025-07-30 18:23:05.467 | INFO     | src.ai_core.vector_store:vector_store:207 - get vector store  : PgVector/my_documents_embeddings_768_fake cache_embeddings: False


In [38]:
vector_store2.add_documents(documents=[Document(page_content="Hello")])
vector_store2.add_documents(documents=[Document(page_content="World")])

['11fb738c-818c-433b-8ceb-2d1bf35b23fb']

In [39]:
  CREATE TABLE products (
      product_id SERIAL PRIMARY KEY,
      name VARCHAR(255) NOT NULL,
      description TEXT,
      price_usd DECIMAL(10, 2) NOT NULL,
      category VARCHAR(255),
      quantity INT DEFAULT 0,
      sku VARCHAR(255) UNIQUE NOT NULL,
      image_url VARCHAR(255),
      metadata JSON,
      embed vector(768) DEFAULT NULL -- vector dimensions depends on the embedding model
  );

SyntaxError: invalid syntax (882427072.py, line 1)